In [0]:
# Preâmbulo para configurar o catálogo
dbutils.widgets.text("catalog", "v_credit_case")
catalog = dbutils.widgets.get("catalog")

bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Catálogo configurado com sucesso!")

In [0]:
# Imports usados nas funções da camada
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
# Função para carregar um DataFrame a partir do nome de uma tabela da silver
def load_df(table_name):
    # Se der erro, a execução deve parar
    try:
        existing = {t.name for t in spark.catalog.listTables(silver_schema)}
    except Exception:
        raise FileNotFoundError(f"O schema {silver_schema} não existe ou não está acessível.")
    if table_name not in existing:
        raise FileNotFoundError(
            f"{table_name} não foi encontrado em {silver_schema}. "
            f"Tabelas disponíveis: {sorted(existing)}"
        )

    return spark.table(f"{silver_schema}.{table_name}")

In [0]:
# Função para salvar o DataFrame na camada gold
def save_df(df, table_name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
            .saveAsTable(f"{gold_schema}.{table_name}")

In [0]:
def add_sequential_key(df, key_name):
    if key_name in df.columns:
        raise ValueError(f"A coluna {key_name} já existe no DataFrame.")

    # Gera a chave única totalmente distribuída e rápida
    keyed = df.withColumn(key_name, F.monotonically_increasing_id())

    # Retorna o DataFrame com a nova chave na primeira posição
    return keyed.select(F.col(key_name), *[F.col(c) for c in df.columns])

In [0]:
# Conta quantas linhas violam a condição esperad
def dq_check(table_name: str, check_name: str, df, condition):
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

In [0]:
# Checagem de qualidade específica para unicidade de chave
def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")